In [ ]:
import json

with open("cvs.json", mode="r") as user_file:
    my_data = json.load(user_file)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql.types import *

spark = SparkSession.\
        builder.\
        appName("pyspark-notebook").\
        master("spark://spark-master:7077").\
        config("spark.executor.memory", "512m").\
        getOrCreate()

In [ ]:
#LOCATION SCHEMA
country_schema = StructType(
    {StructField("capital",StringType(),True),
         StructField("countryId",LongType(),True),
         StructField("countryName",StringType(),True),
         StructField("currency",StringType(),True),
         StructField("iso2",StringType(),True),
         StructField("iso3",StringType(),True),
         StructField("phoneCode",StringType(),True)})

region_schema = StructType({
    StructField("country",country_schema,True),
    StructField("regionCode",StringType(),True),
    StructField("regionId",LongType(),True),
    StructField("regionName",StringType(),True)})

city_schema = StructType({
    StructField("cityId",LongType(),True),
    StructField("cityName",StringType(),True),
    StructField("latitude",DoubleType(),True),
    StructField("longitude",DoubleType(),True),
    StructField("region",region_schema)})

location_schema = StructType([
    StructField("address",StringType(),True),
    StructField("city",city_schema),
    StructField("locationId",LongType(),True),
    StructField("type",StringType(),True)])

#CITIZENSHIP SCHEMA
citizenship_schema = StructType([
    StructField("nationalityId",LongType(),True),
    StructField("nationalityName",StringType(),True)])

#AWARDS SCHEMA
award_schema = StructType([
    StructField("awardId",LongType(),True),
    StructField("title",StringType(),True),
    StructField("awardingInstitution",StringType(),True),
    StructField("dateOfAward",StringType(),True),
    StructField("description",StringType(),True)])

#HOBBIES SCHEMA
hobby_schema = StructType([
    StructField("hobbyId",LongType(),True),
    StructField("name",StringType(),True),
    StructField("description",StringType(),True)])

#PROJECTS SCHEMA
project_schema = StructType([
    StructField("projectId",LongType(),True),
    StructField("name",StringType(),True),
    StructField("fromDate",StringType(),True),
    StructField("toDate",StringType(),True),
    StructField("url",StringType(),True),
    StructField("description",StringType(),True)])

#PHONE NUMEBRS SCHEMA
phone_number_schema = StructType([
    StructField("country",country_schema,True),
    StructField("number",StringType(),True),
    StructField("phoneNumberId",LongType(),True)])

#LANGUAGE SCHEMA
language_schema = StructType({
    StructField("code",StringType(),True),
    StructField("name",StringType(),True),
    StructField("nativeName",StringType(),True),
    StructField("languageId",LongType(),True)})

user_language_schema = StructType([
    StructField("userLanguageId",LongType(),True),
    StructField("listening",StringType(),True),
    StructField("reading",StringType(),True),
    StructField("spokenInteraction",StringType(),True),
    StructField("spokenProduction",StringType(),True),
    StructField("writing",StringType(),True),
    StructField("language",language_schema,True)
])

#EDUCATION SCHEMA
educational_institution_schema = StructType({
    StructField("institutionName",StringType(),True),
    StructField("altInstitutionName",StringType(),True),
    StructField("address",StringType(),True),
    StructField("telephone",StringType(),True),
    StructField("address",StringType(),True),
    StructField("website",StringType(),True),
    StructField("email",StringType(),True),
    StructField("funding",StringType(),True),
    StructField("accreditingAgency",StringType(),True),
    StructField("country",country_schema,True),
    StructField("educationalInstitutionId",LongType(),True)
})

educational_experience = StructType([
    StructField("educationalExperienceId",LongType(),True),
    StructField("specialty",StringType(),True),
    StructField("qualificationTitle",StringType(),True),
    StructField("fromDate",StringType(),True),
    StructField("toDate",StringType(),True),
    StructField("educationalInstitution",educational_institution_schema,True),
])

#PROFESSION SCHEMA
organisation_schema = StructType({
    StructField("organisationId",LongType(),True),
    StructField("organisationName",StringType(),True),
    StructField("address",StringType(),True),
    StructField("city",city_schema,True),
})

professional_experience = StructType([
    StructField("professionalExperienceId",LongType(),True),
    StructField("jobName",StringType(),True),
    StructField("description",StringType(),True),
    StructField("fromDate",StringType(),True),
    StructField("toDate",StringType(),True),
    StructField("organisation",organisation_schema,True),
])

#SKILL SCHEMA
skill_schema = StructType([
    StructField("skillId",LongType(),True),
    StructField("name",StringType(),True),
    StructField("description",StringType(),True),
])

schema = StructType([
	StructField("aboutMe",StringType(),True),
	StructField("birthdate",StringType(),True),
	StructField("canTakeTheIqRequest",BooleanType(),True),
	StructField("cvReferences",StructType({
        StructField("locations", location_schema, True),
        StructField("citizenship", citizenship_schema, True),
        StructField("awards", award_schema, True),
        StructField("hobbies", hobby_schema, True),
        StructField("projects", project_schema, True),
        StructField("awards", award_schema, True),
        StructField("phoneNumbers", phone_number_schema, True),
        StructField("publications", ArrayType(MapType(StringType(),StringType(),True),True),True),
        StructField("iqTests", ArrayType(MapType(StringType(),StringType(),True),True),True),
        StructField("userLanguages", user_language_schema, True),
        StructField("educationalExperiences", educational_experience, True),
        StructField("professionalExperiences", professional_experience, True),
        StructField("userInterpersonalSkills", skill_schema, True),
        StructField("userDigitalSkills", skill_schema, True),
        StructField("userOrganisationalSkills", skill_schema, True),
        StructField("userLeadershipSkills", skill_schema, True),
    }),True),
	StructField("email",StringType(),True),
	StructField("firstname",StringType(),True),
	StructField("gender",StringType(),True),
	StructField("hasImage",BooleanType(),True),
	StructField("iqScore",LongType(),True),
	StructField("lastname",StringType(),True),
	StructField("motto",StringType(),True),
	StructField("requestedRetakeOfIqTest",BooleanType(),True),
	StructField("uid",LongType(),True)])

In [ ]:
import json
df = spark.read.json("cvs.json", multiLine=True)
df.printSchema()

In [ ]:
from pyspark.sql.functions import *

df.withColumn('age', floor(datediff(current_date(), to_date(col('birthdate'), 'yyyy/MM/dd'))/365.25)).collect()

In [ ]:
list(map(lambda item: item.asDict(), df.filter(df.email=="eren@gmail.com").collect()))

In [ ]:
def iterate_nested(s):
    if isinstance(s, (list, tuple)):
        for index, value in enumerate(s):
            yield from iterate_nested(value)
    elif isinstance(s, dict):
        for key, value in s.items():
            yield from iterate_nested(value)
    elif isinstance(s, set):
        for value in s:
            yield from iterate_nested(value)
    elif isinstance(s, str):
        yield s.lower()

list(map(lambda item: item.asDict(), 
         df.withColumn('age', floor(datediff(current_date(), to_date(col('birthdate'), 'yyyy-MM-dd'))/365.25))
         .where((col("age") >= 20) & (col("age") <= 500))
         .rdd
         .filter(lambda x: all([item.lower() in ' '.join(list(iterate_nested(x))) for item in ["ion", "creanga"]]))
         .collect()))


In [ ]:
from itertools import chain
get_item_lambda_fun = lambda x: [item
                                 if type(item) is not dict and type(item) is not list
                                 else get_item_lambda_fun(item.values()) if type(item) is dict
                                 else get_item_lambda_fun(item)
                                 for item in x]

def iterate_nested(s):
    if isinstance(s, (list, tuple)):
        for index, value in enumerate(s):
            yield from iterate_nested(value)
    elif isinstance(s, dict):
        for key, value in s.items():
            yield from iterate_nested(value)
    elif isinstance(s, set):
        for value in s:
            yield from iterate_nested(value)
    elif isinstance(s, str):
        yield s

df.rdd.map(lambda x: ' '.join(list(iterate_nested(x)))).collect()
    

In [ ]:
df.rdd.map(lambda row: row.asDict())

In [ ]:
from pyspark.context import SparkContext


def process_cvs(cvs, keywords=[], min_age=0, max_age=1000):
    df = spark.read.json(spark.sparkContext.parallelize([json.dumps(cvs)]))
    return list(map(lambda item: item.asDict(recursive=True), 
             df.withColumn('age', floor(datediff(current_date(), to_date(col('birthdate'), 'yyyy-MM-dd'))/365.25))
             .where((col("age") >= 20) & (col("age") <= 500))
             .rdd
             .filter(lambda x: all([item.lower() in ' '.join(list(iterate_nested(x))) for item in keywords]))
             .collect()))

#print(json.dumps(my_data))
process_cvs(my_data)

In [ ]:
all([])

In [ ]:
%run -i 'app.py'